
# 15 读懂 WT-plus 工具链与一次失败的优化

> **本章要解决的问题**：筛选器、优化器、追踪系统、原生回放，各自回答什么问题？为什么“优化器说最好”的配置，放到账户里却亏？一次失败的优化，应该怎样读？

## 本章导读

第12章讲了单因子流水线，第14章讲了对账。这一章讲**工具链的分工**与**失败诊断**：

1. 四个角色：筛选、优化、追踪、回放；
2. 试验账本：族、冠军、空仓基准；
3. 消融实验：A/B/AB；
4. 一次失败的优化：数字怎样读。

### 学习目标

- 能说清筛选器、优化器、追踪、回放的分工。
- 能读试验族、冠军和空仓基准。
- 能做 2×2 消融，分离主效应和交互。
- 能诊断“优化成功、实盘失败”的原因。
- 能写一条可复查的失败记录。

### 先修

第12章的因子流水线；第14章的回放对账；第07章的优化。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="15 读懂 WT-plus 工具链与一次失败的优化" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>15 读懂 WT-plus 工具链与一次失败的优化｜知识图</title>
<desc>本章主线：筛选、优化、追踪、回放各答一问；“优化冠军”不等于可上线策略。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a15" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao15" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag15" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">15</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">读懂 WT-plus 工具链与一次失败的优化</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：筛选、优化、追踪、回放各答一问；“优化冠军”不等于可上线策略。</text>
<rect x="41" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="121" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">试验账本与回放结果</text>
<text x="121" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">族、冠军、空仓基准</text>
<rect x="51" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="68" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输入</text>
<rect x="227" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="307" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">2×2 消融实验</text>
<text x="307" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">分离主效应与交互</text>
<rect x="237" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="254" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">处理</text>
<rect x="413" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="493" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">失败诊断与下一步</text>
<text x="493" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">价格 / 数量 / 费用归因</text>
<rect x="423" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="440" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">输出</text>
<rect x="599" y="88" width="160" height="122" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="679" y="141.3" font-size="13.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">可复查记录</text>
<text x="679" y="161.4" font-size="10.5" fill="#6E7B8B" text-anchor="middle">试验次数必须写进结论</text>
<rect x="609" y="97" width="34" height="17" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.1" rx="7"/>
<text x="626" y="109.5" font-size="9.5" fill="#6E7B8B" text-anchor="middle">检查</text>
<path d="M 201 149 L 227 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a15)"/>
<path d="M 387 149 L 413 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a15)"/>
<path d="M 573 149 L 599 149" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a15)"/>
<rect x="41" y="232" width="718" height="152" fill="#FFFFFF" stroke="#E6EFF7" stroke-width="1.5" rx="14"/>
<rect x="58" y="246" width="4" height="11" fill="#E07B39" rx="1"/>
<text x="70" y="256" font-size="11.5" fill="#1B3A5C" font-weight="bold">一次“成功优化”的失败诊断</text>
<text x="58" y="278" font-size="9.5" fill="#6E7B8B">7 组试验：优化器说“冠军”，实盘却转负</text>
<rect x="95" y="322" width="24" height="34" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="107" y="317" font-size="8" fill="#6E7B8B" text-anchor="middle">34</text>
<rect x="135" y="310" width="24" height="46" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="147" y="305" font-size="8" fill="#6E7B8B" text-anchor="middle">46</text>
<rect x="175" y="316" width="24" height="40" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="187" y="311" font-size="8" fill="#6E7B8B" text-anchor="middle">40</text>
<rect x="215" y="294" width="24" height="62" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="227" y="289" font-size="8" fill="#6E7B8B" text-anchor="middle">62</text>
<rect x="255" y="302" width="24" height="54" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="267" y="297" font-size="8" fill="#6E7B8B" text-anchor="middle">54</text>
<rect x="295" y="274" width="24" height="82" fill="#E07B39" stroke="#E07B39" stroke-width="1.2" rx="3"/>
<text x="307" y="269" font-size="8" fill="#6E7B8B" text-anchor="middle">82</text>
<rect x="335" y="306" width="24" height="50" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.2" rx="3"/>
<text x="347" y="301" font-size="8" fill="#6E7B8B" text-anchor="middle">50</text>
<path d="M 95 356 L 400 356" fill="none" stroke="#6E7B8B" stroke-width="1.1" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag15)"/>
<text x="98" y="375" font-size="8.8" fill="#6E7B8B">参数组合 / 试验编号</text>
<text x="213" y="262" font-size="9" fill="#E07B39" text-anchor="middle" font-weight="bold">冠军</text>
<path d="M 310 272 L 430 292" fill="none" stroke="#E07B39" stroke-width="1.3" stroke-linecap="round" stroke-linejoin="round" stroke-dasharray="5 3" marker-end="url(#ao15)"/>
<text x="438" y="289" font-size="9.5" fill="#E07B39" font-weight="bold">实盘 3 个月：净利 −</text>
<text x="438" y="305" font-size="8.8" fill="#6E7B8B">目标函数 ≠ 账户净利</text>
<text x="438" y="322" font-size="8.8" fill="#6E7B8B">只看冠军 → 高估</text>
<text x="438" y="339" font-size="8.8" fill="#6E7B8B">空仓基准要一起算</text>
</svg>

> **图**：本章主线：筛选、优化、追踪、回放各答一问；“优化冠军”不等于可上线策略。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：策略开发、研究员、优化器维护。
- **输入**：试验账本、回放结果、消融配置。
- **输出**：失败诊断、消融表、下一步。
- **在流程的哪一步**：优化与复盘。
- **失败模式**：只看冠军、把“亏得少”当有效、不记试验次数。
- **最小工作流**：读族/冠军/空仓 → 2×2 消融 → 每笔净利 → 诊断 → 记录。

---

## 1. 工具链不是四个角色，是一条流水线

完整的十个工具/护栏见第37章。本章只保留“失败诊断”需要的几类问题：

| 你查什么 | 工具 | 回答 |
|---|---|---|
| 数据/字段够不够 | Great Expectations | 输入是否合格 |
| 哪组参数/表达式最好 | RD-Agent + Optuna | 搜索/超参 |
| 单因子是否值得进组合 | Alphalens-reloaded | IC/分组/换手 |
| 是否只是过拟合 | DSR/PBO | 校正后显著性 |
| 组合绩效/归因 | fincore | 组合层 |
| 真实执行会怎样 | WT-plus 原生回放 | 账户语义 |
| 每轮能否复现 | MLflow | 追踪 |

**这些工具不能互相替代。** 优化器说“好”，只是“在它的目标/样本上最好”；DSR/PBO 才回答“多轮搜索后还显著吗”；回放才回答“账户会怎样”。

**常见错误**：

1. 把“RD-Agent 冠军”当“可上线策略”。
2. 不记录试验族，只看冠军。
3. 不做 DSR/PBO，直接用原始 IC/t 值。
4. 用回放结果反向调参，再宣称“样本外”。

**停下来想一想**：为什么“优化器冠军”常常是过拟合？

<details><summary>对照你的思路</summary>

优化器/LLM 在有限样本上挑最好的一组，等于做了多次比较；噪声也会被挑中。试验次数越多，冠军越可能只是“运气最好”。要看试验族分布、DSR/PBO、样本外、消融。

</details>

---



> **真实代码地图**：`factor_automation_worker.py`（协调）、`FactorToolchain.py`（GE/Optuna/TA-Lib/Alphalens/DSR-PBO/fincore/MLflow）、`factor_compute.py`→`kunquant_runner.py`（计算）、`factor_native_audit.py`（回放审计）。见第37章 §7。

## 2. 试验账本：冠军相对谁

### 2.1 族

同一问题、同一数据、同一目标下的所有试验，叫一个“族”。只看族内最好，不看族内分布，会高估。

### 2.2 空仓基准

冠军的净利要和“空仓（0）”比，也要和“全做/基准策略”比。若全族都亏，冠军只是“亏得少”。

### 2.3 代码：族内冠军



In [2]:

trials = {"a": -8.0, "b": -2.0, "c": -5.0}
winner = max(trials, key=trials.get)
print("族内冠军", winner, "净利", trials[winner])
print("相对空仓", trials[winner] - 0)
print("族内均值", sum(trials.values()) / len(trials))


族内冠军 b 净利 -2.0
相对空仓 -2.0
族内均值 -5.0



**读数与应用**：冠军 b 净利 −2，仍然亏；相对空仓 −2。若只报“b 是冠军”，会误导。

**常见错误**：

1. 只报冠军，不报族内分布。
2. 不和空仓/基准比。
3. 不记录试验次数。

**停下来想一想**：族内冠军 −2、族内均值 −5，说明什么？

<details><summary>对照你的思路</summary>

整个族都在亏，冠军只是“亏得少”；没有证据支持“这组参数能赚钱”。需要换问题/数据/成本假设，而不是继续在族内挑。

</details>

---



## 3. 消融实验：A/B/AB

### 3.1 2×2

同时改两个东西（A、B），测四种组合：

| 配置 | 净利 |
|---|---:|
| base | 10 |
| A | 14 |
| B | 13 |
| AB | 12 |

主效应：

$$\text{A效应}=A-\text{base}=4,\qquad \text{B效应}=B-\text{base}=3.$$

**公式拆解**
- **符号**：base/A/B/AB 为四种配置净利（元，同区间同成本），A 效应、B 效应为主效应。
- **结构**：主效应=只改一项与都不改之差 $A-\text{base}$、$B-\text{base}$；仅当交互为 0 才能相加。
- **代入**：10、14、13、12 → A 效应 4、B 效应 3，可加预测 17。
- **边界**：净利为负时主效应仍可正；交易数不同要改看每笔净利。
- **用途**：消融第一步、判断单项改动；不是因果，也没有误差棒。

交互：

$$\text{AB}-\text{A}-\text{B}+\text{base}=12-14-13+10=-5.$$

**公式拆解**
- **符号**：四个净利（元）同上；交互项 $=\text{AB}-\text{A}-\text{B}+\text{base}$。
- **结构**：差中差，等于实测 AB 减可加预测，也可写 $(\text{AB}-\text{B})-(\text{A}-\text{base})$。
- **代入**：$12-14-13+10=-5$ 元：A、B 单独都好，一起用反而差 5 元。
- **边界**：$=0$ 可加；$<0$ 抵消（争预算/容量），$>0$ 协同但要防凑数。
- **用途**：判断“两个一起上”是否成立；不是显著性，也不能外推。

### 3.2 读法

- 主效应正、交互负：两个改动一起用会互相抵消；
- 只改一个，可能比同时改更好。

### 3.3 代码：消融



In [3]:

net = {"base": 10.0, "A": 14.0, "B": 13.0, "AB": 12.0}
print("A 效应", net["A"] - net["base"])
print("B 效应", net["B"] - net["base"])
print("交互", net["AB"] - net["A"] - net["B"] + net["base"])


A 效应 4.0
B 效应 3.0
交互 -5.0



**读数与应用**：A、B 单独都好，但一起用反而差；交互 −5。报告要写“组合效应”，不能只写单变量。

**常见错误**：

1. 只做单变量，不做组合。
2. 把交互当成“噪声”。
3. 不固定其他条件。

**停下来想一想**：为什么两个“好”的改动一起用会变差？

<details><summary>对照你的思路</summary>

它们可能争用同一个风险预算/容量，或一个改动让另一个的前提失效（比如都减少交易，叠加后样本太少）。交互项就是这种“1+1<2”的度量。

</details>

---



## 4. 一次失败的优化：数字怎样读

### 4.1 一组真实感的试验

| 配置 | 交易数 | 毛利 | 费用 | 基础净利 | 加压净利 |
|---|---:|---:|---:|---:|---:|
| regime_5_20 | 24 | −310 | 147.54 | −457.54 | −570.59 |
| regime_3_15 | 24 | −290 | 147.49 | −437.49 | −548.19 |
| regime_8_30 | 19 | −290 | 116.72 | −406.72 | −491.30 |

### 4.2 读法

- 冠军 regime_8_30 净利 −406.72，仍然亏；
- 交易少、费用低，是“少做少亏”，不是“更会选”；
- 加压更差，说明成本敏感。

### 4.3 代码：失败诊断



In [4]:

import pandas as pd

trials = pd.DataFrame([
    ["regime_5_20", 24, -310, 147.54, -457.54, -570.59],
    ["regime_3_15", 24, -290, 147.49, -437.49, -548.19],
    ["regime_8_30", 19, -290, 116.72, -406.72, -491.30],
], columns=["profile", "trades", "gross", "fees", "base", "stress"])

trials["net_per_trade"] = trials["base"] / trials["trades"]
print(trials.sort_values("base", ascending=False).round(2).to_string(index=False))


    profile  trades  gross   fees    base  stress  net_per_trade
regime_8_30      19   -290 116.72 -406.72 -491.30         -21.41
regime_3_15      24   -290 147.49 -437.49 -548.19         -18.23
regime_5_20      24   -310 147.54 -457.54 -570.59         -19.06



**读数与应用**：每笔净利都在 −17 到 −21 元；冠军只是“交易少”。正确结论是“这批参数/数据不支持盈利”，不是“找到冠军”。

**常见错误**：

1. 把“亏得少”当“有效”。
2. 不看每笔净利。
3. 不看加压成本。

**停下来想一想**：为什么“交易少”会让净利看起来更好？

<details><summary>对照你的思路</summary>

每笔都亏，交易越少，总亏损越小；这是“分母效应”，不是“选择能力”。要看每笔净利、命中率、盈亏比，不能只看总净利。

</details>

---



## 本章小结

### 一张表

| 概念 | 做法 | 用途 |
|---|---|---|
| 工具链 | 数据/搜索/初筛/护栏/组合/回放 | 分工 |
| 试验族 | 同问题/数据/目标 | 防挑冠军 |
| 空仓基准 | 和 0 比 | 防误导 |
| 消融 | base/A/B/AB | 主效应/交互 |
| 每笔净利 | 净利/交易数 | 防分母效应 |

### 八句话

1. 优化器说好，不等于账户会好。
2. 冠军要和族内分布/空仓比。
3. 试验次数要记录。
4. 消融要测组合，不只单变量。
5. 交互项可能为负。
6. 亏得少不是有效。
7. 每笔净利比总净利更诚实。
8. 失败记录要能复查。

---

## 本章习题

**1.** 筛选器/优化器/追踪/回放各回答什么问题？

**2.** 族内冠军 −2、均值 −5，说明什么？

**3.** base 10、A 14、B 13、AB 12。求 A/B 主效应和交互。

**4.** 为什么两个好改动一起用会变差？

**5.** 交易 19 笔、净利 −406.72，每笔多少？

**6.** 为什么“交易少”会让总净利看起来更好？

**7.** 优化器冠军为什么容易过拟合？

**8.** 失败记录至少要写什么？

<details><summary>参考答案</summary>

**1.** 筛选：数据够不够；优化：哪组参数最好；追踪：发生了什么；回放：真实执行会怎样。

**2.** 全族都亏，冠军只是亏得少。

**3.** A 效应 4、B 效应 3、交互 −5。

**4.** 争风险预算/容量，或前提互相破坏。

**5.** $-406.72/19\approx-21.41$ 元/笔。

**6.** 每笔都亏，交易越少总亏损越小。

**7.** 有限样本上多次比较，噪声也会被挑中。

**8.** 数据/参数/成本/试验族/每笔净利/结论。

</details>

---

## 参考资料

- 第07、12、14章。
